<a href="https://colab.research.google.com/github/Fofita911/Fisica-computacional/blob/main/PROYECTO_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Proyecto 3.5.2 Solución de circuitos eléctricos.
# Métodos: Eliminación Gaussiana y Jacobi.
# Nombre: Fierro Gutiérrez Daniela.

import numpy as np

# Construcción de la matriz aumentada A|b (orden n x (n+1))
# Variables: I = [I1,I2,I3,I4,I5,I6]
# Ecuaciones usadas (misma forma que discutimos):
# filas = [eq4, eq3, eq1, eq5, eq2, eq6]  (esto asegura no ceros en diagonal inicial)

def build_augmented(V1, V2):
    # Resistencias y correspondencia ya incorporadas en el planteamiento de A|b
    # Notación: última columna es b
    A_aug = np.array([
        [ 5.0,  5.0,  0.0,  0.0,  0.0,  0.0,   V1    ],  # eq4: R1*I1 + R2*I2 = V1
        [ 0.0,  1.0,  0.0,  1.0,  0.0, -1.0,   0.0   ],  # eq3: I2 + I4 - I6 = 0
        [ 1.0, -1.0, -1.0,  0.0,  0.0,  0.0,   0.0   ],  # eq1: I1 - I3 - I2 = 0
        [ 0.0,  0.0,  0.0, 15.0,  4.0,  0.0,   V2    ],  # eq5: R5*I4 + R4*I5 = V2 (R5=15,R4=4)
        [ 0.0,  0.0,  1.0, -1.0, -1.0,  0.0,   0.0   ],  # eq2: I3 - I5 - I4 = 0
        [ 5.0,  0.0, 10.0,  0.0,  4.0, -4.0,  V1 - V2]   # eq6: R1*I1 + R3*I3 + R4*I5 - R6*I6 = V1-V2
    ], dtype=float)
    return A_aug

# Eliminación Gaussiana manual (con pivoteo parcial)
# Entrada: A_aug (n x (n+1))
# Salida: x (vector solución de tamaño n)

def gauss_elimination_manual(A_aug):
    A = A_aug.copy().astype(float)
    n = A.shape[0]
    # Forward elimination
    for i in range(n):
        # pivoteo parcial: buscar fila con máximo en columna i
        max_row = i + np.argmax(np.abs(A[i:, i]))
        if abs(A[max_row, i]) < 1e-14:
            raise ValueError("Pivote nulo: sistema singular o no tiene solución única.")
        # intercambiar
        if max_row != i:
            A[[i, max_row], :] = A[[max_row, i], :]
        # normalizar fila pivote (no estrictamente necesario, pero claro)
        pivot = A[i, i]
        A[i, :] = A[i, :] / pivot
        # eliminar por debajo
        for j in range(i+1, n):
            factor = A[j, i]
            A[j, :] = A[j, :] - factor * A[i, :]
    # Back substitution
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = A[i, -1] - np.dot(A[i, i+1:n], x[i+1:n])
    return x

# Iteraciones de Jacobi
# Entrada: A (n x n), b (n), x0, tol, max_iter
# Salida: x, converged_flag, iterations

def jacobi_method(A, b, x0=None, tol=1e-8, max_iter=5000):
    n = A.shape[0]
    if x0 is None:
        x0 = np.zeros(n)
    x = x0.copy()
    D = np.diag(A)
    if np.any(np.isclose(D, 0.0)):
        raise ValueError("A tiene ceros en la diagonal; Jacobi no válido sin reordenar.")
    R = A - np.diagflat(D)
    for k in range(1, max_iter+1):
        x_new = (b - R.dot(x)) / D
        # criterio relativo
        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            return x_new, True, k
        x = x_new
    return x, False, max_iter


# Función para armar A y b desde A_aug

def split_A_b(A_aug):
    A = A_aug[:, :-1]
    b = A_aug[:, -1]
    return A, b


# Casos pedidos
cases = [(20.0, 10.0), (20.0, 50.0), (20.0, -20.0)]

for V1, V2 in cases:
    print("\n---------------------------------------------")
    print(f"Resolviendo para V1 = {V1} V, V2 = {V2} V")
    A_aug = build_augmented(V1, V2)
    # Eliminación Gaussiana
    x_gauss = gauss_elimination_manual(A_aug)
    print("\nSolución (Eliminación Gaussiana):")
    for i, val in enumerate(x_gauss, start=1):
        print(f"I{i} = {val:.8f} A")
    # Intento Jacobi (sobre A y b)
    A, b = split_A_b(A_aug)
    try:
        x0 = np.zeros(A.shape[0])
        x_jacobi, conv, iters = jacobi_method(A, b, x0=x0, tol=1e-10, max_iter=50000)
        if conv:
            print(f"\nJacobi convergió en {iters} iteraciones. Solución:")
            for i, val in enumerate(x_jacobi, start=1):
                print(f"I{i} = {val:.8f} A")
        else:
            print("\nJacobi NO convergió (llegó a iter. máxima).")
    except Exception as e:
        print("\nJacobi no se pudo ejecutar:", e)

    # Indicar corrientes que cambiaron de signo
    print("\nCambios de sentido (según signos de la solución por Gauss):")
    for i, val in enumerate(x_gauss, start=1):
        signo = "mismo sentido" if val >= 0 else "cambió de sentido"
        print(f"I{i}: {val:.8f} A → {signo}")


---------------------------------------------
Resolviendo para V1 = 20.0 V, V2 = 10.0 V

Solución (Eliminación Gaussiana):
I1 = 2.35668790 A
I2 = 1.64331210 A
I3 = 0.71337580 A
I4 = 0.64968153 A
I5 = 0.06369427 A
I6 = 2.29299363 A

Jacobi NO convergió (llegó a iter. máxima).

Cambios de sentido (según signos de la solución por Gauss):
I1: 2.35668790 A → mismo sentido
I2: 1.64331210 A → mismo sentido
I3: 0.71337580 A → mismo sentido
I4: 0.64968153 A → mismo sentido
I5: 0.06369427 A → mismo sentido
I6: 2.29299363 A → mismo sentido

---------------------------------------------
Resolviendo para V1 = 20.0 V, V2 = 50.0 V

Solución (Eliminación Gaussiana):
I1 = 2.10191083 A
I2 = 1.89808917 A
I3 = 0.20382166 A
I4 = 4.47133758 A
I5 = -4.26751592 A
I6 = 6.36942675 A

Jacobi NO convergió (llegó a iter. máxima).

Cambios de sentido (según signos de la solución por Gauss):
I1: 2.10191083 A → mismo sentido
I2: 1.89808917 A → mismo sentido
I3: 0.20382166 A → mismo sentido
I4: 4.47133758 A → mismo s